In [1]:
import seaborn             as sns
import libraries.utilities as sul
import os

# Assuming the `bin` directory is in the user's home directory
# For sqs
home_directory = os.path.expanduser("~")
bin_directory = os.path.join(home_directory, "bin")
os.environ["PATH"] += os.pathsep + bin_directory

from pymatgen.core.surface   import generate_all_slabs
from pymatgen.core.structure import Structure
from pymatgen.io.ase         import AseAtomsAdaptor
from ase.io.vasp             import write_vasp

sns.set_theme()

We only consider coherent interfaces (which has matching lattices from both sides of the interface, meaning that there is a repeat along the interface that lets build periodicity along the interface surface).

In [2]:
# Define name of folder and path to reference POSCAR
general_folder = 'input/CeO2'
path_to_POSCAR = 'POSCAR-uc-CeO2'

# Maximum number for the Miller index in each direction
max_index = 4

# Whether to repair terminations with broken bonds or just omit them
repair = True

# Minimum thicknesses for slab and vacuum
min_slab_size   = 20.0  # Angstroms
min_vacuum_size = 20.0  # Angstroms

The surface formation energy ($E_{surface}$) can be expressed in terms of the area ($S$) of the surface, the energy ($E_{slab}$) of the slab and the energy of the bulk system ($E_{bulk}$) as:

\begin{equation}
    E_{surface} = \frac{E_{slab} - E_{bulk}}{2 S}
\end{equation}

where all energies are per atom.

In [3]:
# Move POSCAR to defect folder
# The folder is named as Bi2S3_x_y_z_vi (eg, Bi2S3_0_1_0_v0)
i = 0
while True:
    if not os.path.exists(general_folder):
        # Generate new folder
        os.system(f'mkdir {general_folder}')
    
    slab_folder = f'{general_folder}/slab_v{i}'
    if not os.path.exists(slab_folder):
        # Generate new folder
        os.system(f'mkdir {slab_folder}')

        # Copy POSCAR there (named as unticell, and POSCAR for creating supercell)
        os.system(f'cp {path_to_POSCAR} {slab_folder}/POSCAR')
        break
    i +=1
slab_folder

'input/CeO2/slab_v0'

# ML-IAP relaxation

In [4]:
# Unit-cell relaxation

# Define paths to bulk and POSCAR
path_to_bulk = f'{slab_folder}/bulk'

# Generate directory for current slab
os.system(f'mkdir {path_to_bulk}')

# Copy POSCAR to new directory
os.system(f'cp {slab_folder}/POSCAR {path_to_bulk}/POSCAR')

os.system(f'cp {slab_folder}/POSCAR {path_to_bulk}/CONTCAR')

# Read relaxed structure
structure = Structure.from_file(f'{path_to_bulk}/CONTCAR')

# Save slab information
sul.save_json(structure, data='bulk', filename=f'{path_to_bulk}/slab_data.json')

/Users/cibran/work/UPC/SlabOptimization/.venv/lib/python3.13/site-packages/pymatgen/core/structure.py:3087: EncodingWarning: We strongly encourage explicit `encoding`, and we would use UTF-8 by default as per PEP 686
  with zopen(filename, mode="rt", errors="replace") as file:


# Slab generation

In [ ]:
# Generate all slabs
slabs = generate_all_slabs(structure,
                           max_index=max_index,
                           min_slab_size=min_slab_size,
                           min_vacuum_size=min_vacuum_size,
                           repair=repair)

In [6]:
for n, slab in enumerate(slabs):
    print(n, "Polar:", slab.is_polar(), "Symmetric: ", slab.is_symmetric())

0 Polar: False Symmetric:  False
1 Polar: False Symmetric:  True
2 Polar: False Symmetric:  True
3 Polar: False Symmetric:  False
4 Polar: False Symmetric:  False
5 Polar: False Symmetric:  True
6 Polar: False Symmetric:  False


Surface formation energy of shape $S$ for slab of energy $E_S$ with $N$ formula units and $E_{bulk}$ bulk energy is:

\begin{equation}
    E_S = \frac{E_S - E_{bulk}}{2 S}
\end{equation}

where all energies are per atom.

In [7]:
# Initialize the data dictionary for storing all slab energy calculations
# Iterate over slabs
for i, slab in enumerate(slabs):
    print()
    print(f'Slab {i+1}')
    print()
    print(f'Miller index: {slab.miller_index}')
    print(f'Shift: {slab.shift:.2g}')
    print(f'Surface area: {slab.surface_area:.2g}')
    print(f'Number of sites: {len(slab.sites)}')

    # Miller index tuple to string
    miller_index_str = '_'.join(str(element) for element in slab.miller_index)
    miller_index_str = f'{miller_index_str}_i_{i}'

    # Define current folder
    miller_folder = f'{slab_folder}/{miller_index_str}'

    # Generate new folder
    os.system(f'mkdir {miller_folder}')

    # Save slab structure into miller_folder
    write_vasp(f'{miller_folder}/POSCAR', AseAtomsAdaptor.get_atoms(slab), direct=True, sort=True)

    # Save slab information
    sul.save_json(slab, filename=f'{miller_folder}/slab_data.json')


Slab 1

Miller index: (1, 1, 1)
Shift: 0.13
Surface area: 13
Number of sites: 21

Slab 2

Miller index: (1, 1, 1)
Shift: 0.5
Surface area: 13
Number of sites: 21

Slab 3

Miller index: (1, 1, 0)
Shift: 0.25
Surface area: 21
Number of sites: 36

Slab 4

Miller index: (1, 0, 0)
Shift: 0.12
Surface area: 15
Number of sites: 24

Slab 5

Miller index: (2, 2, 1)
Shift: 0.063
Surface area: 44
Number of sites: 72

Slab 6

Miller index: (2, 1, 1)
Shift: 0.13
Surface area: 36
Number of sites: 60

Slab 7

Miller index: (2, 1, 0)
Shift: 0.062
Surface area: 33
Number of sites: 60
